# Collected Data Validation

## Project: AI-Assisted Pull Request Outcome Prediction

In this step, we validate the collected AIDev datasets by checking their quality, consistency, completeness, and reliability before using them for data cleaning, integration, analysis, and machine learning.

In [1]:
# Import libraries required for data validation

import pyarrow.parquet as pq
import pandas as pd

In [2]:
# Open the original AIDev Parquet dataset for validation

file_path = "../data/raw/all_pull_request.parquet"

file = pq.ParquetFile(file_path)

print("Number of rows:", file.metadata.num_rows)
print("Number of columns:", len(file.schema.names))

Number of rows: 2743854
Number of columns: 14


In [3]:
# Check missing values in each column

null_counts = {}

for i in range(file.num_row_groups):
    df_group = file.read_row_group(i).to_pandas()
    
    for column in df_group.columns:
        null_counts[column] = null_counts.get(column, 0) + df_group[column].isnull().sum()

null_counts = pd.Series(null_counts)

print(null_counts)

id                 0
number             0
title              1
body           21481
agent              0
user_id            0
user               0
state              0
created_at         0
closed_at     292402
merged_at     533798
repo_id         8902
repo_url           0
html_url           0
dtype: int64


In [4]:
# Check for duplicate Pull Request records

duplicate_count = 0

for i in range(file.num_row_groups):
    df_group = file.read_row_group(i).to_pandas()
    duplicate_count += df_group.duplicated().sum()

print("Duplicate rows found:", duplicate_count)

Duplicate rows found: 0


In [5]:
# Check the data type of each column

for i in range(file.num_row_groups):
    df_group = file.read_row_group(i).to_pandas()
    
    if i == 0:
        print(df_group.dtypes)

id              int64
number          int64
title             str
body              str
agent             str
user_id         int64
user              str
state             str
created_at        str
closed_at         str
merged_at         str
repo_id       float64
repo_url          str
html_url          str
dtype: object


In [6]:
# Check the unique values in the Pull Request state column

state_counts = {}

for i in range(file.num_row_groups):
    df_group = file.read_row_group(i, columns=["state"]).to_pandas()
    counts = df_group["state"].value_counts(dropna=False)
    
    for state, count in counts.items():
        state_counts[state] = state_counts.get(state, 0) + count

print(state_counts)

{'closed': 2451452, 'open': 292402}


In [7]:
# Check whether Pull Request IDs are unique

id_counts = {}

for i in range(file.num_row_groups):
    df_group = file.read_row_group(i, columns=["id"]).to_pandas()
    
    for pr_id in df_group["id"]:
        id_counts[pr_id] = id_counts.get(pr_id, 0) + 1

duplicate_ids = sum(count > 1 for count in id_counts.values())

print("Duplicate PR IDs:", duplicate_ids)

Duplicate PR IDs: 0


In [8]:
# Check the relationship between state, closed_at, and merged_at

outcome_counts = {}

for i in range(file.num_row_groups):
    df_group = file.read_row_group(
        i,
        columns=["state", "closed_at", "merged_at"]
    ).to_pandas()

    df_group["closed_at_missing"] = df_group["closed_at"].isna()
    df_group["merged_at_missing"] = df_group["merged_at"].isna()

    counts = df_group.groupby(
        ["state", "closed_at_missing", "merged_at_missing"]
    ).size()

    for combination, count in counts.items():
        outcome_counts[combination] = (
            outcome_counts.get(combination, 0) + count
        )

for combination, count in outcome_counts.items():
    print(combination, ":", count)

('closed', False, False) : 2210056
('closed', False, True) : 241396
('open', True, True) : 292402


In [9]:
# Check the date columns for valid and invalid date values

date_columns = ["created_at", "closed_at", "merged_at"]

for i in range(file.num_row_groups):
    df_group = file.read_row_group(i, columns=date_columns).to_pandas()

    for column in date_columns:
        converted = pd.to_datetime(df_group[column], errors="coerce")
        invalid = converted.isna() & df_group[column].notna()

        print(f"Row Group {i} - {column}: {invalid.sum()} invalid dates")

Row Group 0 - created_at: 0 invalid dates
Row Group 0 - closed_at: 0 invalid dates
Row Group 0 - merged_at: 0 invalid dates
Row Group 1 - created_at: 0 invalid dates
Row Group 1 - closed_at: 0 invalid dates
Row Group 1 - merged_at: 0 invalid dates
Row Group 2 - created_at: 0 invalid dates
Row Group 2 - closed_at: 0 invalid dates
Row Group 2 - merged_at: 0 invalid dates


In [10]:
# Check whether Pull Request dates follow a logical chronological order

date_order_errors = {
    "closed_before_created": 0,
    "merged_before_created": 0,
    "merged_after_closed": 0
}

for i in range(file.num_row_groups):
    df_group = file.read_row_group(
        i,
        columns=["created_at", "closed_at", "merged_at"]
    ).to_pandas()

    created = pd.to_datetime(df_group["created_at"], errors="coerce")
    closed = pd.to_datetime(df_group["closed_at"], errors="coerce")
    merged = pd.to_datetime(df_group["merged_at"], errors="coerce")

    date_order_errors["closed_before_created"] += (closed < created).sum()
    date_order_errors["merged_before_created"] += (merged < created).sum()
    date_order_errors["merged_after_closed"] += (merged > closed).sum()

print(date_order_errors)

{'closed_before_created': np.int64(0), 'merged_before_created': np.int64(0), 'merged_after_closed': np.int64(50)}


In [11]:
# Find the 50 Pull Requests with an unusual date order

date_errors = []

for i in range(file.num_row_groups):
    df_group = file.read_row_group(
        i,
        columns=["id", "created_at", "closed_at", "merged_at"]
    ).to_pandas()

    created = pd.to_datetime(df_group["created_at"], errors="coerce")
    closed = pd.to_datetime(df_group["closed_at"], errors="coerce")
    merged = pd.to_datetime(df_group["merged_at"], errors="coerce")

    errors = df_group[merged > closed].copy()
    date_errors.append(errors)

date_errors = pd.concat(date_errors, ignore_index=True)

print("Records with date-order issue:", len(date_errors))
date_errors.head(50)

Records with date-order issue: 50


,id,created_at,closed_at,merged_at
0,3338498648,2025-08-20T15:01:22Z,2025-08-20T15:05:00Z,2025-08-20T15:10:28Z
1,3382047981,2025-09-04T04:03:01Z,2025-09-04T06:29:52Z,2025-09-04T06:29:55Z
2,3521453239,2025-10-16T11:23:55Z,2025-10-16T11:42:57Z,2025-10-16T11:42:58Z
3,3540468626,2025-10-22T11:27:14Z,2025-10-22T11:45:02Z,2025-10-22T11:45:04Z
4,3519282037,2025-10-15T19:21:21Z,2025-10-15T20:19:19Z,2025-10-15T21:02:30Z
5,3472006404,2025-10-01T04:51:18Z,2025-10-01T05:18:09Z,2025-10-01T05:18:12Z
6,3176607539,2025-06-25T18:40:02Z,2025-06-25T18:46:40Z,2025-06-25T18:46:42Z
7,3496753851,2025-10-08T20:14:11Z,2025-10-09T03:13:58Z,2025-10-09T03:14:02Z
8,3293951306,2025-08-05T17:51:42Z,2025-08-05T17:56:03Z,2025-08-05T17:56:16Z
9,3443629185,2025-09-23T04:27:17Z,2025-09-23T14:01:12Z,2025-09-23T14:01:14Z


In [12]:
# Check the AI agents and their Pull Request counts

agent_counts = {}

for i in range(file.num_row_groups):
    df_group = file.read_row_group(i, columns=["agent"]).to_pandas()
    counts = df_group["agent"].value_counts(dropna=False)

    for agent, count in counts.items():
        agent_counts[agent] = agent_counts.get(agent, 0) + count

print(agent_counts)

{'OpenAI_Codex': 2069595, 'Copilot': 349695, 'Google_Jules': 50490, 'Devin': 43298, 'Claude_Code': 18232, 'Cursor': 212544}


In [13]:
# Check repository ID values and missing repository IDs

repo_id_counts = {}

for i in range(file.num_row_groups):
    df_group = file.read_row_group(i, columns=["repo_id"]).to_pandas()
    
    print(f"Row Group {i} - Missing repo_id:", df_group["repo_id"].isna().sum())
    print(f"Row Group {i} - Unique repo_id:", df_group["repo_id"].nunique())

Row Group 0 - Missing repo_id: 2903
Row Group 0 - Unique repo_id: 197648
Row Group 1 - Missing repo_id: 3668
Row Group 1 - Unique repo_id: 118171
Row Group 2 - Missing repo_id: 2331
Row Group 2 - Unique repo_id: 133805


In [14]:
# Check repository URL values and missing repository URLs

for i in range(file.num_row_groups):
    df_group = file.read_row_group(i, columns=["repo_url"]).to_pandas()
    
    print(f"Row Group {i} - Missing repo_url:", df_group["repo_url"].isna().sum())
    print(f"Row Group {i} - Unique repo_url:", df_group["repo_url"].nunique())

Row Group 0 - Missing repo_url: 0
Row Group 0 - Unique repo_url: 198033
Row Group 1 - Missing repo_url: 0
Row Group 1 - Unique repo_url: 118443
Row Group 2 - Missing repo_url: 0
Row Group 2 - Unique repo_url: 134145


In [15]:
# Check empty text values in Pull Request title and body

empty_counts = {
    "title_empty": 0,
    "body_empty": 0
}

for i in range(file.num_row_groups):
    df_group = file.read_row_group(
        i,
        columns=["title", "body"]
    ).to_pandas()

    empty_counts["title_empty"] += (
        df_group["title"].fillna("").str.strip().eq("").sum()
    )

    empty_counts["body_empty"] += (
        df_group["body"].fillna("").str.strip().eq("").sum()
    )

print(empty_counts)

{'title_empty': np.int64(1), 'body_empty': np.int64(21481)}


In [16]:
# Check user ID values and missing user IDs

for i in range(file.num_row_groups):
    df_group = file.read_row_group(i, columns=["user_id"]).to_pandas()

    print(
        f"Row Group {i} - Missing user_id:",
        df_group["user_id"].isna().sum()
    )

    print(
        f"Row Group {i} - Unique user_id:",
        df_group["user_id"].nunique()
    )

Row Group 0 - Missing user_id: 0
Row Group 0 - Unique user_id: 69139
Row Group 1 - Missing user_id: 0
Row Group 1 - Unique user_id: 82631
Row Group 2 - Missing user_id: 0
Row Group 2 - Unique user_id: 99713


In [17]:
# Check username values for missing or empty entries

for i in range(file.num_row_groups):
    df_group = file.read_row_group(i, columns=["user"]).to_pandas()

    empty_users = (
        df_group["user"].fillna("").str.strip().eq("").sum()
    )

    print(f"Row Group {i} - Empty user values:", empty_users)

Row Group 0 - Empty user values: 0
Row Group 1 - Empty user values: 0
Row Group 2 - Empty user values: 0


In [18]:
# Check whether Pull Request URLs contain valid GitHub links

invalid_urls = 0

for i in range(file.num_row_groups):
    df_group = file.read_row_group(
        i,
        columns=["html_url"]
    ).to_pandas()

    invalid_urls += (
        ~df_group["html_url"]
        .fillna("")
        .str.startswith("https://github.com/")
    ).sum()

print("Invalid GitHub PR URLs:", invalid_urls)

Invalid GitHub PR URLs: 0


----

In [19]:
# Open the AIDev repository and user datasets for validation.

repository_path = "../data/raw/all_repository.parquet"
user_path = "../data/raw/all_user.parquet"

repository_file = pq.ParquetFile(repository_path)
user_file = pq.ParquetFile(user_path)

print("Repository rows:", repository_file.metadata.num_rows)
print("Repository columns:", len(repository_file.schema.names))

print("\nUser rows:", user_file.metadata.num_rows)
print("User columns:", len(user_file.schema.names))

Repository rows: 326798
Repository columns: 8

User rows: 161255
User columns: 5


In [20]:
# Check missing values in the repository and user datasets.

repository_df = pd.read_parquet(repository_path)
user_df = pd.read_parquet(user_path)

print("Repository missing values:")
print(repository_df.isnull().sum())

print("\nUser missing values:")
print(user_df.isnull().sum())

Repository missing values:
id                0
url               0
license      221769
full_name         0
is_forked         0
language      44681
forks             0
stars             0
dtype: int64

User missing values:
id            50
login          0
followers     50
following     50
created_at    50
dtype: int64


In [21]:
# Check whether repository IDs and user IDs contain duplicates.

repository_duplicate_ids = repository_df["id"].duplicated().sum()
user_duplicate_ids = user_df["id"].duplicated().sum()

print("Duplicate repository IDs:", repository_duplicate_ids)
print("Duplicate user IDs:", user_duplicate_ids)

Duplicate repository IDs: 0
Duplicate user IDs: 49


In [22]:
# Inspect the user records with duplicate IDs.

duplicate_user_ids = user_df.loc[
    user_df["id"].notna() & user_df["id"].duplicated(keep=False)
].sort_values("id")

print("Duplicate user ID records:")
print(duplicate_user_ids)

Duplicate user ID records:
Empty DataFrame
Columns: [id, login, followers, following, created_at]
Index: []


In [23]:
# Check the data types of repository and user columns.

print("Repository data types:")
print(repository_df.dtypes)

print("\nUser data types:")
print(user_df.dtypes)

Repository data types:
id             int64
url              str
license          str
full_name        str
is_forked       bool
language         str
forks        float64
stars        float64
dtype: object

User data types:
id            float64
login             str
followers     float64
following     float64
created_at        str
dtype: object


In [24]:
# Check whether repository and user numeric values contain invalid negative values.

print("Repository negative forks:", (repository_df["forks"] < 0).sum())
print("Repository negative stars:", (repository_df["stars"] < 0).sum())

print("\nUser negative followers:", (user_df["followers"] < 0).sum())
print("User negative following:", (user_df["following"] < 0).sum())

Repository negative forks: 0
Repository negative stars: 0

User negative followers: 0
User negative following: 0


In [26]:
# Check whether raw PR repository and user IDs match the repository and user datasets.

pr_path = "../data/raw/all_pull_request.parquet"

pr_ids_df = pd.read_parquet(pr_path, columns=["repo_id", "user_id"])

pr_repo_ids = set(pr_ids_df["repo_id"].dropna().unique())
repository_ids = set(repository_df["id"].dropna().unique())

pr_user_ids = set(pr_ids_df["user_id"].dropna().unique())
user_ids = set(user_df["id"].dropna().unique())

print("PR repository IDs not found in repository dataset:", len(pr_repo_ids - repository_ids))
print("PR user IDs not found in user dataset:", len(pr_user_ids - user_ids))

PR repository IDs not found in repository dataset: 0
PR user IDs not found in user dataset: 49


In [27]:
# Count how many PR records use the 49 user IDs that are missing from the user dataset.

unmatched_user_ids = pr_user_ids - user_ids

affected_prs = pr_ids_df["user_id"].isin(unmatched_user_ids).sum()

print("Unmatched user IDs:", len(unmatched_user_ids))
print("PR records affected:", affected_prs)

Unmatched user IDs: 49
PR records affected: 344975


In [28]:
# Check whether user account creation dates contain invalid or missing values.

user_created_at = pd.to_datetime(user_df["created_at"], errors="coerce", utc=True)

print("Missing/invalid user created_at:", user_created_at.isna().sum())
print("Future user created_at:", (user_created_at > pd.Timestamp.now(tz="UTC")).sum())

Missing/invalid user created_at: 50
Future user created_at: 0


---

## Validation Conclusion

The AIDev datasets were successfully validated before the data cleaning stage.

### Pull Request Dataset
- Contains **2,743,854 rows** and **14 columns**.
- Missing values, duplicates, data types, states, PR IDs, dates, agents, repository IDs, text values, and GitHub URLs were validated.
- **50 date-order anomalies** were identified and retained for later handling.
- Missing `repo_id` values were identified but the corresponding PR URLs are valid.

### Repository Dataset
- Contains **326,798 repositories** and **8 columns**.
- Missing values were identified in `license` and `language`.
- Repository IDs contain **0 valid duplicates**.
- Fork and star values contain **no negative values**.
- All PR repository IDs have matching repository records.

### User Dataset
- Contains **161,255 user records** and **5 columns**.
- 50 records have missing user metadata.
- User IDs contain **0 valid duplicates**; the apparent 49 duplicates were caused by missing IDs.
- Followers and following values contain **no negative values**.
- 50 `created_at` values are missing, with **0 future dates**.
- 49 user IDs referenced by PRs are not present in the user dataset, affecting **344,975 PR records**.

The original raw AIDev datasets remain unchanged. The identified data quality and relationship issues will be handled during the Data Cleaning and Data Integration stages.

---